# Fashion-MNIST Clothing Classifier
**Project P21 | Programming for AI | BS Artificial Intelligence**

This notebook builds a beginner-friendly image classification pipeline for Fashion-MNIST.
The model predicts which of 10 clothing categories appears in a 28×28 grayscale image.

**Project approach**
- Explore and visualize the dataset.
- Prepare image pixels for machine learning.
- Train a small neural network using a PyTorch training loop written in this notebook.
- Train a Scikit-learn `MLPClassifier` as a comparison model.
- Evaluate both models and inspect commonly confused clothing categories.

> This is an educational prototype. It is not intended for commercial or safety-critical use.

## 1. Install and import libraries

If any import fails, install the missing package in your notebook environment, then restart the kernel.

In [ ]:
# Run this cell if the required libraries are not installed:
# %pip install torch torchvision scikit-learn matplotlib seaborn pandas numpy

import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", DEVICE)

## 2. Load Fashion-MNIST

The dataset is downloaded automatically the first time this cell runs. It contains 28×28 grayscale images.
To keep this coursework experiment manageable, we use a reproducible subset of 10,000 training images and 2,000 test images.

In [ ]:
DATA_DIR = "../data"

transform = transforms.ToTensor()

train_full = datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform
)
test_full = datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform
)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

train_count = 10000
test_count = 2000

train_indices = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(SEED))[:train_count]
test_indices = torch.randperm(len(test_full), generator=torch.Generator().manual_seed(SEED))[:test_count]

train_subset = torch.utils.data.Subset(train_full, train_indices.tolist())
test_subset = torch.utils.data.Subset(test_full, test_indices.tolist())

print("Training images:", len(train_subset))
print("Testing images:", len(test_subset))
print("Image shape:", train_full[0][0].shape)
print("Number of classes:", len(class_names))

## 3. Explore the data

First, display a few images with their correct labels. Then check how many examples of each category are present in our selected training subset.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_subset[i]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.suptitle("Sample Fashion-MNIST Images")
plt.tight_layout()
plt.show()

In [ ]:
train_labels = np.array([label for _, label in train_subset])
class_counts = pd.Series(train_labels).value_counts().sort_index()

plt.figure(figsize=(10, 4))
sns.barplot(x=class_names, y=[class_counts.get(i, 0) for i in range(10)])
plt.title("Class Distribution in Training Subset")
plt.xlabel("Clothing category")
plt.ylabel("Number of images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "Class": class_names,
    "Training images": [class_counts.get(i, 0) for i in range(10)]
}))

## 4. Prepare features

Each image has 28 × 28 = 784 pixel values. We flatten the image into a 784-value feature vector.
`ToTensor()` has already scaled pixel values to the range 0 to 1.

In [ ]:
def make_arrays(dataset):
    images = torch.stack([image for image, _ in dataset])
    labels = torch.tensor([label for _, label in dataset], dtype=torch.long)
    features = images.view(images.shape[0], -1)
    return features, labels

X_train_tensor, y_train_tensor = make_arrays(train_subset)
X_test_tensor, y_test_tensor = make_arrays(test_subset)

print("Training feature shape:", tuple(X_train_tensor.shape))
print("Testing feature shape:", tuple(X_test_tensor.shape))
print("Pixel value range:", float(X_train_tensor.min()), "to", float(X_train_tensor.max()))

## 5. Build a small neural network with PyTorch

The network has an input layer of 784 values, one hidden layer with 128 neurons, and an output layer with 10 class scores.
The training loop below explicitly performs the forward pass, loss calculation, backpropagation, and optimizer update.

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


torch_model = FashionMLP().to(DEVICE)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.001)

BATCH_SIZE = 128
EPOCHS = 5

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

epoch_losses = []
start_time = time.perf_counter()

for epoch in range(EPOCHS):
    torch_model.train()
    total_loss = 0.0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        # 1. Clear gradients from the previous batch
        optimizer.zero_grad()

        # 2. Forward pass: calculate predicted class scores
        predictions = torch_model(batch_features)

        # 3. Calculate how far predictions are from the true labels
        loss = loss_function(predictions, batch_labels)

        # 4. Backward pass: calculate gradients
        loss.backward()

        # 5. Update model weights
        optimizer.step()

        total_loss += loss.item() * batch_features.size(0)

    average_loss = total_loss / len(train_dataset)
    epoch_losses.append(average_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {average_loss:.4f}")

torch_training_time = time.perf_counter() - start_time
print(f"PyTorch training time: {torch_training_time:.2f} seconds")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses, marker="o")
plt.title("PyTorch Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.grid(True)
plt.show()

## 6. Evaluate the PyTorch model

We calculate overall accuracy, per-class accuracy, and a confusion matrix. A confusion matrix shows which categories the model mixes up.

In [ ]:
torch_model.eval()
with torch.no_grad():
    test_scores = torch_model(X_test_tensor.to(DEVICE))
    torch_predictions = test_scores.argmax(dim=1).cpu().numpy()

y_test = y_test_tensor.numpy()
torch_accuracy = accuracy_score(y_test, torch_predictions)

print(f"PyTorch test accuracy: {torch_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(
    y_test, torch_predictions, target_names=class_names, zero_division=0
))

In [ ]:
torch_cm = confusion_matrix(y_test, torch_predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(
    torch_cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names
)
plt.title("PyTorch Confusion Matrix")
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
per_class_accuracy = []
for class_id, class_name in enumerate(class_names):
    mask = (y_test == class_id)
    accuracy = (torch_predictions[mask] == y_test[mask]).mean()
    per_class_accuracy.append(accuracy)

class_accuracy_table = pd.DataFrame({
    "Class": class_names,
    "PyTorch accuracy": per_class_accuracy
}).sort_values("PyTorch accuracy", ascending=False)

display(class_accuracy_table)

## 7. Train a Scikit-learn MLPClassifier for comparison

We use the same training and test subsets. `MLPClassifier` is Scikit-learn's multi-layer perceptron classifier.

In [ ]:
X_train = X_train_tensor.numpy()
X_test = X_test_tensor.numpy()
y_train = y_train_tensor.numpy()

sklearn_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=30,
    random_state=SEED,
    early_stopping=True
)

start_time = time.perf_counter()
sklearn_model.fit(X_train, y_train)
sklearn_training_time = time.perf_counter() - start_time

sklearn_predictions = sklearn_model.predict(X_test)
sklearn_accuracy = accuracy_score(y_test, sklearn_predictions)

print(f"Scikit-learn MLP test accuracy: {sklearn_accuracy:.4f}")
print(f"Scikit-learn training time: {sklearn_training_time:.2f} seconds")

In [ ]:
comparison = pd.DataFrame({
    "Model": ["PyTorch MLP", "Scikit-learn MLPClassifier"],
    "Test accuracy": [torch_accuracy, sklearn_accuracy],
    "Training time (seconds)": [torch_training_time, sklearn_training_time]
})
display(comparison)

## 8. Inspect commonly confused categories

The off-diagonal cells of a confusion matrix show incorrect predictions. The largest off-diagonal value identifies a pair of categories that the model confuses most often.

In [ ]:
torch_cm_without_diagonal = torch_cm.copy()
np.fill_diagonal(torch_cm_without_diagonal, 0)

actual_id, predicted_id = np.unravel_index(
    np.argmax(torch_cm_without_diagonal),
    torch_cm_without_diagonal.shape
)

print("Most confused actual category:", class_names[actual_id])
print("Most common incorrect prediction:", class_names[predicted_id])
print(
    "Number of such mistakes:",
    int(torch_cm_without_diagonal[actual_id, predicted_id])
)

In [ ]:
# Display a few examples where the PyTorch model made an incorrect prediction.
wrong_indices = np.where(torch_predictions != y_test)[0][:10]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, index in zip(axes.flat, wrong_indices):
    ax.imshow(X_test[index].reshape(28, 28), cmap="gray")
    ax.set_title(
        f"True: {class_names[y_test[index]]}\nPred: {class_names[torch_predictions[index]]}",
        fontsize=8
    )
    ax.axis("off")
plt.suptitle("Examples of Misclassified Images")
plt.tight_layout()
plt.show()

## 9. Try a prediction on one image

Change `sample_index` to inspect a different image from the test subset.

In [ ]:
sample_index = 0

sample_image = X_test_tensor[sample_index]
with torch.no_grad():
    sample_scores = torch_model(sample_image.unsqueeze(0).to(DEVICE))
    sample_prediction = int(sample_scores.argmax(dim=1).item())

plt.imshow(sample_image.reshape(28, 28), cmap="gray")
plt.title(
    f"Actual: {class_names[int(y_test[sample_index])]} | "
    f"Predicted: {class_names[sample_prediction]}"
)
plt.axis("off")
plt.show()

## 10. Project findings and limitations

Complete this short interpretation after running all cells:
- Which model achieved higher test accuracy?
- Which clothing categories were most difficult to classify?
- Which two categories were most commonly confused?
- Did training time differ between the two models?

**Limitations:** This experiment uses a subset of Fashion-MNIST and small fully connected networks. The images are small, grayscale and centered. Real-world clothing photos may have different backgrounds, lighting, angles and image sizes. The model should therefore be treated as a learning demonstration, not a reliable real-world clothing recognition service.

## Dataset and references

- Dataset: Fashion-MNIST by Zalando Research.
- Dataset repository: https://github.com/zalandoresearch/fashion-mnist
- Dataset paper: https://arxiv.org/abs/1708.07747
- Libraries: PyTorch, torchvision, Scikit-learn, NumPy, Pandas, Matplotlib and Seaborn.

**Reproducibility:** The random seed is set to 42. The notebook uses a fixed subset of 10,000 training and 2,000 test images.